# 🎬 Wan2.2 免费视频生成 — Colab 版
**零成本 GPU · 文生视频 · 图生视频**

使用方式：
1. 菜单栏 → 代码执行程序 → 更改运行时类型 → 选 **T4 GPU**
2. 按顺序运行每个单元格（点击 ▶ 按钮）
3. 生成的视频自动下载到本地
4. 导入剪映剪辑 → 发布抖音

## 📦 第一步：安装依赖（约 3 分钟）

In [ ]:
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q diffusers transformers accelerate imageio[ffmpeg] opencv-python
!pip install -q huggingface_hub

print("✅ 依赖安装完成")

## 🎯 第二步：加载 Wan2.2 模型
选择模型大小：
- **T2V-1.3B** (推荐) — 显存要求低，T4 可跑
- **T2V-14B** — 质量更高，需 A100

In [ ]:
import torch
from diffusers import WanPipeline
from diffusers.utils import export_to_video
import datetime

# 检查 GPU
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "无GPU"
gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1e9 if torch.cuda.is_available() else 0
print(f"🔧 GPU: {gpu_name} ({gpu_mem:.1f} GB)")

# T4 选 1.3B, A100 选 14B
model_id = "Wan-AI/Wan2.2-T2V-1.3B" if gpu_mem < 20 else "Wan-AI/Wan2.2-T2V-14B"
print(f"📦 加载模型: {model_id}")

pipe = WanPipeline.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    variant="fp16",
)
pipe.to("cuda")
print("✅ 模型加载完成！")

## ✍️ 第三步：输入提示词，生成视频
每次运行生成一个 5 秒视频（约 2-3 分钟）

In [ ]:
# ====== 在这里改你的提示词 ======
PROMPT = "一只橘猫在阳光下的窗台上打哈欠，4K，电影质感，柔光"

# 生成参数
NUM_FRAMES = 81  # 81帧 ≈ 5秒 (15fps)
GUIDANCE_SCALE = 5.0

print(f"🎬 开始生成: {PROMPT}")

output = pipe(
    prompt=PROMPT,
    num_frames=NUM_FRAMES,
    guidance_scale=GUIDANCE_SCALE,
    height=720,
    width=720,
    generator=torch.Generator("cuda").manual_seed(42),
)

timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
filename = f"wan22_{timestamp}.mp4"
export_to_video(output.frames[0], filename, fps=15)
print(f"✅ 视频已生成: {filename}")
print("  → 导入剪映 → 调色/加字幕/配乐 → 发布抖音")

## 🔄 第四步（可选）：图生视频
先上传图片，再用 Wan2.2 动起来

In [ ]:
from google.colab import files
from PIL import Image

# 上传图片
uploaded = files.upload()
image_path = list(uploaded.keys())[0]
image = Image.open(image_path).convert("RGB")
print(f"📷 已上传: {image_path} ({image.size})")

# 图生视频
from diffusers import WanImageToVideoPipeline

i2v_pipe = WanImageToVideoPipeline.from_pretrained(
    "Wan-AI/Wan2.2-I2V-1.3B",
    torch_dtype=torch.float16,
    variant="fp16",
)
i2v_pipe.to("cuda")

output = i2v_pipe(
    image=image,
    prompt="轻柔的运动，自然流畅",
    num_frames=81,
    guidance_scale=5.0,
    generator=torch.Generator("cuda").manual_seed(42),
)

timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
filename = f"wan22_i2v_{timestamp}.mp4"
export_to_video(output.frames[0], filename, fps=15)
print(f"✅ 图生视频已生成: {filename}")

## 📤 第五步：下载视频到本地
生成完成后自动触发下载

In [ ]:
# 列出所有生成的视频
import glob, os
videos = sorted(glob.glob("*.mp4"), key=os.path.getmtime, reverse=True)
print("📂 生成的视频:")
for v in videos[:10]:
    size_mb = os.path.getsize(v) / 1e6
    print(f"   {v} — {size_mb:.1f} MB")

# 下载最新视频
if videos:
    from google.colab import files
    files.download(videos[0])
    print(f"📥 开始下载: {videos[0]}")
else:
    print("⚠️ 还没有生成视频，请先运行上面的单元格")

---
## 💡 提示词参考

```
# 风景类
"雪山日出，云雾缓缓流动，镜头向前推进，4K 超清，电影质感"

# 科技类
"未来城市夜景，霓虹灯闪烁，飞行汽车穿梭，赛博朋克风格，竖屏"

# 美食类
"拉面师傅手工拉面，面粉飞散，慢动作，暖色调，食欲感"

# 萌宠类
"金毛犬在海边奔跑，夕阳逆光，慢镜头，温馨治愈"
```